In [1]:
import sys
sys.path.insert(0, '../lib')

In [2]:
import collections
import functools
import joblib
import os
import pathlib

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.stats.multitest

import common_data

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200
%config InlineBackend.figure_format = "retina"

In [ ]:
ROOT = common_data.DATA
BASE = ROOT / '05_pseudobulk/30a_pathogens_coarse'

In [ ]:
adata = sc.read_h5ad(common_data.SC_NORM)

In [6]:
sc_labels = pd.read_csv(common_data.SC_LABELS, index_col=0)

In [7]:
sc_labels.perturbation_groups_2.unique()

array(['discard', 'Gram-*', 'Late SARS-CoV-2; Gram+',
       'Early SARS-CoV-2; Gram+', 'Early SARS-CoV-2', 'Late SARS-CoV-2',
       'NPC', 'Pseudomonas aeruginosa; SARS-CoV-2', 'Gram+',
       'Gram-*; Gram+', 'Pseudomonas aeruginosa', 'Healthy'], dtype=object)

In [8]:
sc_labels['pathogens_coarse'] = sc_labels.perturbation_groups_2.replace({
    'Gram-*': 'Bacteria',
    'Gram+': 'Bacteria',
    'Pseudomonas aeruginosa': 'Bacteria',
    'Gram-*; Gram+': 'Bacteria',
    'Late SARS-CoV-2; Gram+': 'Mixed',
    'Early SARS-CoV-2; Gram+': 'Mixed',
    'Pseudomonas aeruginosa; SARS-CoV-2': 'Mixed',
})

In [9]:
sc_labels.pathogens_coarse.value_counts()

discard             115
Bacteria             58
Early SARS-CoV-2     35
Mixed                33
NPC                  26
Late SARS-CoV-2      25
Healthy               9
Name: pathogens_coarse, dtype: int64

In [10]:
bal_to_pathogen = sc_labels[
    ['bal_barcode', 'pathogens_coarse']
].set_index('bal_barcode').pathogens_coarse

In [11]:
idx = adata.obs.bal_barcode.isin(bal_to_pathogen.index)
adata.obs.loc[idx, 'pathogens_coarse'] = bal_to_pathogen[adata.obs.bal_barcode[idx]].values

In [12]:
groups = sc_labels.pathogens_coarse.unique().tolist()

In [13]:
groups.remove('discard')

In [14]:
%%time
PSEUDOBULK_CELLS = 50
PSEUDOBULK_EXPR_IN_GROUP = 0.8
genes_to_keep = {}
for ct in adata.obs.Level_6.unique():
    genes = None
    for group in groups:
        samples = adata.obs.bal_barcode[adata.obs.pathogens_coarse.eq(group)].unique()
        pseudobulks = []
        for sample in samples:
            idx = adata.obs.Level_6.eq(ct) & adata.obs.bal_barcode.eq(sample)
            if idx.sum() < PSEUDOBULK_CELLS:
                continue
            pseudobulks.append(adata.raw.X[idx, :].sum(axis=0).A1)
        if len(pseudobulks) == 0:
            continue
        pseudobulks = pd.DataFrame(pseudobulks, columns=adata.raw.var_names)
        expr_frac = (pseudobulks > 0).sum(axis=0) / pseudobulks.shape[0]
        group_genes = pseudobulks.columns[expr_frac.ge(PSEUDOBULK_EXPR_IN_GROUP)]
        if genes is None:
            genes = group_genes.to_numpy()
        else:
            genes = np.union1d(genes, group_genes)
    genes_to_keep[ct] = genes

CPU times: user 1min 1s, sys: 0 ns, total: 1min 1s
Wall time: 1min 1s


In [15]:
{k: len(v) for k, v in genes_to_keep.items() if v is not None}

{'CD4 T cells': 10694,
 'CD8 T cells': 10761,
 'Mast cells': 9966,
 'NUPR1+ Macs': 14031,
 'B cells': 10204,
 'MRC1+C1QA+': 12255,
 'DC2': 10996,
 'MRC1+C1QA-': 12183,
 'Proliferating CD4 T cells': 11039,
 'Proliferating CD8 T cells': 10921,
 'Classical monocytes-2 IL1B': 10227,
 'Secretory cells': 15135,
 'Proliferating NUPR1+ Macs': 12054,
 'Tregs': 9679,
 'DC1': 11388,
 'Ciliated cells': 13243,
 'gdT cells': 9878,
 'Migratory DC': 11759,
 'Interstitial macrophages': 10265,
 'Hematopoietic stem cells': 16004,
 'Classical monocytes-1 CCR2': 9680,
 'pDC': 11292,
 'AT1 and AT2': 16358,
 'Proliferating plasma cells': 12121,
 'Non-classical monocytes': 10699,
 'Plasma cells': 11125,
 'Proliferating gdT cells': 11823}

In [16]:
genes_to_keep['Perivascular macrophages'] = genes_to_keep['Interstitial macrophages']

In [ ]:
PADJ_CUTOFF = 0.05
class ComparisonInfo:
    def __init__(self, control, condition, genes, genes_to_keep):
        self.control = control
        self.condition = condition
        self.genes_raw = genes
        self.filter_genes(genes_to_keep)

    def filter_genes(self, genes_to_keep):
        filtered_degs = self.genes_raw.loc[self.genes_raw.index.isin(genes_to_keep), :].copy()
        filtered_degs = filtered_degs.loc[filtered_degs.padj.notna()].copy()
        # recompute FDR correction on the filtered genes:
        filtered_degs['padj'] = statsmodels.stats.multitest.fdrcorrection(
            filtered_degs.pvalue,
            alpha=PADJ_CUTOFF
        )[1]
        # recompute gene status based on new `padj`
        filtered_degs['sign'] = ''
        filtered_degs.loc[
            filtered_degs.padj.lt(PADJ_CUTOFF)
            & filtered_degs.log2FoldChange.gt(0),
            'sign'
        ] = f'Up in {self.condition}'
        filtered_degs.loc[
            filtered_degs.padj.lt(PADJ_CUTOFF)
            & filtered_degs.log2FoldChange.lt(0),
            'sign'
        ] = f'Up in {self.control}'
        self.genes = filtered_degs


class CellTypeInfo:
    def __init__(self, path, task_info, genes_to_keep):
        self.path = path
        self.task_info = task_info
        self.comparisons = []
        self.meta = pd.read_csv(path / 'meta.csv', index_col=0)
        self.name = self.meta.cell_type.values[0]

        self.load_comparisons(genes_to_keep)

    def load_comparisons(self, genes_to_keep):
        for run in self.path.glob('**/degs.csv'):
            self.comparisons.append(
                ComparisonInfo(
                    self.task_info.column_values[0],
                    self.task_info.column_values[1],
                    pd.read_csv(run, index_col=0),
                    genes_to_keep[self.name]
                )
            )

    @property
    def n_comparisons(self):
        return len(self.comparisons)

In [18]:
class TaskData:
    def __init__(self, task, task_info):
        self.task = task
        self.task_info = task_info
        self.info = {}

In [19]:
SHORTCUTS = {
    'Early SARS-CoV-2': 'eCOVID',
    'Late SARS-CoV-2': 'lCOVID',
    'Bacteria': 'Bac',
    'Mixed': 'Mix',
    'Healthy': 'H',
    'NPC': 'N'
}

In [20]:
# manually define
tasks = [
    ('Healthy', 'Early SARS-CoV-2'),
    ('Healthy', 'Late SARS-CoV-2'),
    ('Healthy', 'Bacteria'),
    ('Healthy', 'Mixed'),
    ('Healthy', 'NPC'),
    ('NPC', 'Early SARS-CoV-2'),
    ('NPC', 'Late SARS-CoV-2'),
    ('NPC', 'Bacteria'),
    ('NPC', 'Mixed'),
    ('Early SARS-CoV-2', 'Late SARS-CoV-2'),
    ('Early SARS-CoV-2', 'Bacteria'),
    ('Early SARS-CoV-2', 'Mixed'),
    ('Late SARS-CoV-2', 'Bacteria'),
    ('Late SARS-CoV-2', 'Mixed'),
    ('Bacteria', 'Mixed')
]

In [21]:
%%time
data = {}
for task in tasks:
    task_name = f'{task[0]}_vs_{task[1]}'
    task_info = common_data.TaskInfo(
        pathname=f'{SHORTCUTS[task[0]]}_vs_{SHORTCUTS[task[1]]}',
        column='pathogens_coarse',
        column_values=task,
        split_column=None
    )
    task_data = TaskData(task, task_info)
    for cell_type_path in sorted((BASE / task_info.pathname).iterdir()):
        if cell_type_path.name.startswith('.') or cell_type_path.name.startswith('_'):
            continue
        if not cell_type_path.is_dir():
            continue
        if not (cell_type_path / 'meta.csv').exists():
            continue
        info = CellTypeInfo(cell_type_path, task_info, genes_to_keep)
        if info.n_comparisons > 0:
            task_data.info[cell_type_path.name] = info
    if len(task_data.info) > 0:
        data[task_name] = task_data

CPU times: user 4.26 s, sys: 0 ns, total: 4.26 s
Wall time: 12.3 s


In [22]:
data

{'Healthy_vs_Early SARS-CoV-2': <__main__.TaskData at 0x1509b6765c60>,
 'Healthy_vs_Late SARS-CoV-2': <__main__.TaskData at 0x1509b60d1810>,
 'Healthy_vs_Bacteria': <__main__.TaskData at 0x1507d26e65f0>,
 'Healthy_vs_Mixed': <__main__.TaskData at 0x1507d1c033d0>,
 'Healthy_vs_NPC': <__main__.TaskData at 0x1507d1017b20>,
 'NPC_vs_Early SARS-CoV-2': <__main__.TaskData at 0x1507cfd79000>,
 'NPC_vs_Late SARS-CoV-2': <__main__.TaskData at 0x1507cf48ba30>,
 'NPC_vs_Bacteria': <__main__.TaskData at 0x1507ce2afeb0>,
 'NPC_vs_Mixed': <__main__.TaskData at 0x1507ccdff7c0>,
 'Early SARS-CoV-2_vs_Late SARS-CoV-2': <__main__.TaskData at 0x1507c1506f50>,
 'Early SARS-CoV-2_vs_Bacteria': <__main__.TaskData at 0x1507aadf1c60>,
 'Early SARS-CoV-2_vs_Mixed': <__main__.TaskData at 0x1507a9ae6f20>,
 'Late SARS-CoV-2_vs_Bacteria': <__main__.TaskData at 0x1507a81b5600>,
 'Late SARS-CoV-2_vs_Mixed': <__main__.TaskData at 0x1507a6eab580>,
 'Bacteria_vs_Mixed': <__main__.TaskData at 0x1507a5c87f10>}

In [ ]:
joblib.dump(data, '31b_deg_data.joblib')

Save filtered DEGs as csv to run GSEA on them

In [ ]:
BASE = ROOT / '05_pseudobulk/30b_degs'
for _, task in data.items():
    for k, ct_info in task.info.items():
        comp = ct_info.comparisons[0]
        deg_path = BASE / task.task_info.pathname / k / 'degs.csv'
        if deg_path.exists():
            print(f'File {deg_path} already exists, skipping')
            continue
        # Threshold GSEA analysis to at least 1000 genes in comparison
        if comp.genes.shape[0] < 1000:
            continue
        ct_path = BASE / task.task_info.pathname / k
        os.makedirs(ct_path, exist_ok=True)
        comp.genes.sort_values('log2FoldChange').to_csv(deg_path)